In [ ]:
!pip install langchain openai gradio gtts langchain_openai langchain_community

INFO: pip is looking at multiple versions of typer to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.1 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: click
    Found existing installation: click 8.3.1
    Uninstalling click-8.3.1:
      Successfully uninstalled click-8.3.1
  A

In [ ]:
!pip install openai-whisper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 17.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=b7a7c231bdd83852ece02347864484b7893e41ba52856ea20dc330d262d52589
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper


In [ ]:
!pip install ffmpeg-python

In [ ]:
chat_history = []

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
import gradio as gr
from gtts import gTTS
import whisper
import os

In [ ]:
# whisper_model = whisper.load_model("medium")

In [ ]:
os.environ["OPENAI_API_KEY"] = "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"
os.environ["OPENAI_API_BASE"] = "https://openrouter.ai/api/v1"
llm = ChatOpenAI(
    temperature=0.7,
    model="gpt-4o-mini"
)

In [ ]:
def get_personality(personality, language):

    if personality == "Funny":
        return "اتكلم بطريقة مرحة وخفيفة وخلّي الكلام ممتع" if language=="Arabic" else "Be fun and playful"

    elif personality == "Calm":
        return "اتكلم بهدوء واطمّن الطفل وخليه يحس بالأمان" if language=="Arabic" else "Be calm and reassuring"

    return ""

In [ ]:
def build_prompt(user_type, language, personality):

    personality_text = get_personality(personality, language)

    if user_type == "Child":
        base = "طفل عنده ADHD، خلي الكلام بسيط وقصير" if language=="Arabic" else "Child with ADHD, keep it simple and short"
    else:
        base = "ولي أمر لطفل ADHD، قدم نصائح عملية" if language=="Arabic" else "Parent of ADHD child, give practical advice"

    system_msg = f"{base}. {personality_text}"

    prompt = ChatPromptTemplate.from_messages([
        ("system", system_msg),
        ("placeholder", "{history}"),
        ("user", "{input}")
    ])

    return prompt

In [ ]:
def detect_emotion(text):

    prompt = f"""
حدد حالة الطفل من النص:
(سعيد - متوتر - مشتت - زهقان)

النص: {text}
"""

    result = llm.invoke(prompt)
    return result.content

In [ ]:
def generate_exercise(emotion, language):

    if language == "Arabic":
        prompt = f"اقترح تمرين بسيط لطفل ADHD حالته {emotion}"
    else:
        prompt = f"Suggest a simple exercise for a child with ADHD who is {emotion}"

    result = llm.invoke(prompt)
    return result.content

In [ ]:
points = 0

def reward_system():
    global points
    points += 10
    return f"⭐ نقاطك: {points}"

In [ ]:
emotion_log = []

def update_dashboard(emotion):
    emotion_log.append(emotion)
    return f"سجل الحالات: {emotion_log[-5:]}"

In [ ]:
def clean_text(text, language):

    if language == "Arabic":
        prompt = f"""
النص التالي خارج من تحويل صوتي وبه أخطاء.
صححه وخليه عربي مفهوم بدون تغيير المعنى:

{text}
"""
    else:
        prompt = f"""
The following text is from speech recognition and contains errors.
Fix it and make it clear without changing the meaning:

{text}
"""

    result = llm.invoke(prompt)
    return result.content

In [ ]:
import whisper
import ffmpeg

whisper_model = whisper.load_model("base")

def speech_to_text(audio_path):

    try:
        # نحول الصوت لـ wav مضبوط
        output_path = "clean_audio.wav"

        (
            ffmpeg
            .input(audio_path)
            .output(output_path, format='wav', acodec='pcm_s16le', ac=1, ar='16000')
            .run(overwrite_output=True)
        )

        # نحوله لنص
        result = whisper_model.transcribe(output_path)

        return result["text"]

    except Exception as e:
        return f"Error in STT: {str(e)}"

100%|████████████████████████████████████████| 139M/139M [00:00<00:00, 214MiB/s]


In [ ]:
def text_to_speech(text, language):
    lang = "ar" if language == "Arabic" else "en"
    tts = gTTS(text=text, lang=lang)
    file_path = "response.mp3"
    tts.save(file_path)
    return file_path

In [ ]:
def full_pipeline(text_input, audio_input, user_type, language, personality):

    global chat_history, points

    # صوت → نص
    if audio_input is not None:
        user_input = speech_to_text(audio_input)
    else:
        user_input = text_input

    if audio_input:
      raw_text = speech_to_text(audio_input)
      user_input = clean_text(raw_text, language)
    # Emotion
    emotion = detect_emotion(user_input)

    # Exercise
    exercise = generate_exercise(emotion, language)

    # Reward
    reward = reward_system()

    # Dashboard
    dashboard = update_dashboard(emotion)

    # History
    history_text = ""
    for h in chat_history:
        history_text += f"User: {h['user']}\nBot: {h['bot']}\n"

    personality_text = get_personality(personality, language)

    system_msg = f"""
أنت مساعد لطفل ADHD.
حالته الحالية: {emotion}
{personality_text}
"""

    full_prompt = f"""
{system_msg}

{history_text}

User: {user_input}
"""

    response = llm.invoke(full_prompt)
    bot_reply = response.content

    chat_history.append({"user": user_input, "bot": bot_reply})

    audio = text_to_speech(bot_reply, language)

    final_text = f"""
🧠 الحالة: {emotion}

💬 الرد:
{bot_reply}

🎯 تمرين:
{exercise}

🏆 {reward}

📊 {dashboard}
"""

    return user_input, final_text, audio

In [ ]:
interface = gr.Interface(
    fn=full_pipeline,
    inputs=[
        gr.Textbox(label="اكتب أو اتكلم"),
        gr.Audio(sources=["upload", "microphone"], type="filepath"),
        gr.Radio(["Child", "Parent"]),
        gr.Radio(["Arabic", "English"]),
        gr.Radio(["Funny", "Calm"])
    ],
    outputs=[
        gr.Textbox(label="Input", lines=2),
        gr.Textbox(label="Output", lines=15),
        gr.Audio()
    ],
    title="🧠 ADHD Smart Assistant"
)

interface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3287e673802f2b5bef.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


#chatbot writing , voice-not-arabic



In [ ]:
# ============================================
# 1. تثبيت المكتبات المطلوبة
# ============================================
!pip install langchain openai gradio gtts langchain_openai langchain_community openai-whisper ffmpeg-python

# ============================================
# 2. الاستيرادات
# ============================================
from langchain_openai import ChatOpenAI
import gradio as gr
from gtts import gTTS
import whisper
import ffmpeg
import os
import tempfile

# ============================================
# 3. إعداد المفاتيح والنماذج
# ============================================
os.environ["OPENAI_API_KEY"] = "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"
os.environ["OPENAI_API_BASE"] = "https://openrouter.ai/api/v1"

llm = ChatOpenAI(temperature=0.7, model="gpt-4o-mini")
whisper_model = whisper.load_model("base")

# ============================================
# 4. المتغيرات العامة
# ============================================
points = 0
emotion_log = []

# ============================================
# 5. دوال المساعدة
# ============================================
def get_personality(personality, language):
    if personality == "Funny":
        return "اتكلم بطريقة مرحة وخفيفة وخلّي الكلام ممتع" if language == "Arabic" else "Be fun and playful, use humor"
    elif personality == "Calm":
        return "اتكلم بهدوء واطمّن الطفل وخليه يحس بالأمان" if language == "Arabic" else "Be calm and reassuring"
    return ""

def detect_emotion(text):
    prompt = f"""حدد حالة الطفل من النص:
(سعيد - متوتر - مشتت - زهقان)

النص: {text}

أكتب الحالة فقط بكلمة واحدة:"""

    result = llm.invoke(prompt)
    return result.content.strip()

def generate_exercise(emotion, language):
    if language == "Arabic":
        prompt = f"اقترح تمرين بسيط وسريع لطفل ADHD حالته {emotion} (جملة واحدة قصيرة)"
    else:
        prompt = f"Suggest a simple quick exercise for a child with ADHD who is {emotion} (one short sentence)"

    result = llm.invoke(prompt)
    return result.content

def speech_to_text(audio_path):
    """تحويل الصوت لنص باستخدام Whisper"""
    if audio_path is None:
        return None

    try:
        # إنشاء ملف مؤقت للصوت النظيف
        with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp_file:
            output_path = tmp_file.name

        # تحويل الصوت لـ wav بصيغة مناسبة
        (
            ffmpeg
            .input(audio_path)
            .output(output_path, format='wav', acodec='pcm_s16le', ac=1, ar='16000')
            .run(overwrite_output=True, quiet=True)
        )

        # التعرف على النص
        result = whisper_model.transcribe(output_path)

        # حذف الملف المؤقت
        os.unlink(output_path)

        return result["text"]
    except Exception as e:
        return f"خطأ في تحويل الصوت: {str(e)}"

def text_to_speech(text, language):
    """تحويل النص لصوت"""
    lang = "ar" if language == "Arabic" else "en"
    tts = gTTS(text=text, lang=lang)

    with tempfile.NamedTemporaryFile(suffix=".mp3", delete=False) as tmp_file:
        file_path = tmp_file.name

    tts.save(file_path)
    return file_path

def reward_system():
    global points
    points += 10
    return f"⭐ نقاطك: {points}"

def update_dashboard(emotion):
    global emotion_log
    emotion_log.append(emotion)
    # نحتفظ بآخر 5 مشاعر فقط
    if len(emotion_log) > 5:
        emotion_log.pop(0)
    return f"📊 آخر المشاعر: {', '.join(emotion_log)}"

# ============================================
# 6. دالة الرد الرئيسية
# ============================================
def respond(text_input, audio_input, history, user_type, language, personality):
    """
    text_input: النص المدخل
    audio_input: مسار ملف الصوت (من المايك)
    history: تاريخ المحادثة
    """
    global points

    # ===== الخطوة 1: معالجة الإدخال (نص أو صوت) =====
    if audio_input is not None:
        # لو فيه صوت من المايك
        user_input = speech_to_text(audio_input)
        if user_input is None:
            user_input = ""
        elif user_input.startswith("خطأ"):
            error_msg = user_input
            history.append(("🎤 [محاولة صوتية]", f"⚠️ {error_msg}"))
            return history, None, "", None
    else:
        # لو إدخال نص
        user_input = text_input.strip() if text_input else ""

    if not user_input:
        return history, None, "", None

    # ===== الخطوة 2: تحليل المشاعر =====
    emotion = detect_emotion(user_input)

    # ===== الخطوة 3: اقتراح تمرين =====
    exercise = generate_exercise(emotion, language)

    # ===== الخطوة 4: تحديث النقاط =====
    reward = reward_system()

    # ===== الخطوة 5: تحديث لوحة المعلومات =====
    dashboard = update_dashboard(emotion)

    # ===== الخطوة 6: بناء الشخصية =====
    personality_text = get_personality(personality, language)

    # تحديد نوع المستخدم
    if user_type == "Child":
        user_context = "طفل عنده ADHD، خلي الكلام بسيط وقصير" if language == "Arabic" else "Child with ADHD, keep it simple and short"
    else:
        user_context = "ولي أمر لطفل ADHD، قدم نصائح عملية" if language == "Arabic" else "Parent of ADHD child, give practical advice"

    # ===== الخطوة 7: بناء الرد من الـ LLM =====
    system_msg = f"{user_context}. {personality_text}\n\nحالة الطفل الحالية: {emotion}"

    # بناء تاريخ المحادثة
    history_text = ""
    for h in history[-5:]:  # آخر 5 محادثات بس عشان نوفر tokens
        history_text += f"المستخدم: {h[0]}\nالمساعد: {h[1]}\n"

    full_prompt = f"""{system_msg}

تاريخ المحادثة:
{history_text}

المستخدم: {user_input}

المساعد:"""

    response = llm.invoke(full_prompt)
    bot_reply = response.content

    # ===== الخطوة 8: إضافة التمرين والمكافأة للرد =====
    if language == "Arabic":
        final_reply = f"""{bot_reply}

---
🎯 **تمرين مقترح:** {exercise}

{reward}
{dashboard}
"""
    else:
        final_reply = f"""{bot_reply}

---
🎯 **Suggested Exercise:** {exercise}

{reward}
{dashboard}
"""

    # ===== الخطوة 9: تحويل الرد لصوت =====
    audio_file = text_to_speech(bot_reply, language)

    # ===== الخطوة 10: إضافة للتاريخ =====
    # نضيف الرسالة للتاريخ
    if audio_input is not None:
        user_display = f"🎤 {user_input}"
    else:
        user_display = user_input

    history.append((user_display, final_reply))

    return history, audio_file, "", None  # نرجع None عشان نفضي المدخلات

# ============================================
# 7. واجهة المستخدم (Live Chatbot مع مايك)
# ============================================
with gr.Blocks(title="🧠 ADHD Smart Assistant", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # 🧠 مساعد ADHD الذكي
    ### 💬 اكتب أو 🎙️ استخدم المايك للتحدث - المساعد هيحلل مشاعرك ويقترح تمارين مناسبة!
    """)

    # إعدادات المستخدم
    with gr.Row():
        with gr.Column(scale=1):
            user_type = gr.Radio(
                choices=["Child", "Parent"],
                label="👤 أنا",
                value="Child"
            )
            language = gr.Radio(
                choices=["Arabic", "English"],
                label="🌐 اللغة",
                value="Arabic"
            )
            personality = gr.Radio(
                choices=["Funny", "Calm"],
                label="🎭 الشخصية",
                value="Funny"
            )

    # شات بوكس الدردشة الحية
    chatbot = gr.Chatbot(
        label="💬 المحادثة",
        height=500,
        bubble_full_width=False,
        avatar_images=(None, "🧠")
    )

    # صندوق الإدخال (نص)
    text_input = gr.Textbox(
        label="✏️ اكتب رسالتك هنا",
        placeholder="اكتب رسالتك...",
        lines=2,
        scale=4
    )

    # مايك للتسجيل المباشر
    audio_input = gr.Audio(
        sources=["microphone"],
        type="filepath",
        label="🎙️ اضغط على المايك وتكلم",
        scale=1
    )

    with gr.Row():
        send_btn = gr.Button("📤 إرسال", variant="primary", scale=1)
        clear_btn = gr.Button("🗑️ مسح المحادثة", variant="secondary", scale=1)

    # صندوق الصوت للتشغيل
    audio_output = gr.Audio(
        label="🔊 الرد الصوتي",
        type="filepath",
        visible=True
    )

    # ===== ربط الأحداث =====

    # دالة لتجميع المدخلات والرد
    def process_and_respond(text, audio, history, user_type, language, personality):
        return respond(text, audio, history, user_type, language, personality)

    # عند الضغط على زر الإرسال
    send_btn.click(
        process_and_respond,
        inputs=[text_input, audio_input, chatbot, user_type, language, personality],
        outputs=[chatbot, audio_output, text_input, audio_input]
    )

    # عند الضغط على Enter في صندوق النص
    text_input.submit(
        process_and_respond,
        inputs=[text_input, audio_input, chatbot, user_type, language, personality],
        outputs=[chatbot, audio_output, text_input, audio_input]
    )

    # عند تسجيل صوت (auto-submit)
    audio_input.stop_recording(
        process_and_respond,
        inputs=[text_input, audio_input, chatbot, user_type, language, personality],
        outputs=[chatbot, audio_output, text_input, audio_input]
    )

    # مسح المحادثة
    clear_btn.click(
        lambda: ([], None, "", None),
        None,
        [chatbot, audio_output, text_input, audio_input]
    )

# ============================================
# 8. تشغيل التطبيق
# ============================================
if __name__ == "__main__":
    demo.launch(share=True, debug=True)

/tmp/ipykernel_5492/2409820016.py:220: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="🧠 ADHD Smart Assistant", theme=gr.themes.Soft()) as demo:
/tmp/ipykernel_5492/2409820016.py:246: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(
/tmp/ipykernel_5492/2409820016.py:246: DeprecationWarning: The 'bubble_full_width' parameter will be removed in Gradio 6.0. This parameter no longer has any effect.
  chatbot = gr.Chatbot(
/tmp/ipykernel_5492/2409820016.py:246: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://d67e4d162b5c285694.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7862 <> https://d67e4d162b5c285694.gradio.live


#live chatbot 2

In [ ]:
!pip install --upgrade huggingface_hub gradio whisper openai-whisper langchain_openai gtts

In [ ]:
# ============================================
# 1. تثبيت المكتبات المطلوبة (تشغل مرة واحدة)
# ============================================

import gradio as gr
import whisper
import tempfile
import os
from gtts import gTTS
from langchain_openai import ChatOpenAI

# ==============================
# 🔑 إعداد الـ API (تأكد من وضع مفتاحك هنا)
# ==============================
# Replace 'YOUR_OPENROUTER_KEY' with your actual OpenRouter API Key
os.environ["OPENAI_API_KEY"] = "YOUR_OPENROUTER_KEY"
os.environ["OPENAI_API_BASE"] = "https://openrouter.ai/api/v1"

# تحميل الموديلات
# استخدمنا "tiny" ليكون أخف وأسرع في المعالجة
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
whisper_model = whisper.load_model("tiny")

# ==============================
# 📊 المتغيرات العامة (تصفير مع كل تشغيل)
# ==============================
points = 0
emotion_log = []

# ==============================
# 🧠 المنطق البرمجي (Backend)
# ==============================

def detect_emotion(text):
    prompt = f"حدد الحالة الشعورية للطفل من الجملة التالية (سعيد - متوتر - مشتت - زهقان) بكلمة واحدة فقط: {text}"
    try:
        return llm.invoke(prompt).content.strip()
    except:
        return "غير محدد"

def generate_exercise(emotion):
    prompt = f"اقترح تمرين ذهني أو حركي بسيط جداً وقصير لطفل ADHD يشعر بـ {emotion}. (جملة واحدة)"
    try:
        return llm.invoke(prompt).content
    except:
        return "خذ نفساً عميقاً واسترخِ قليلاً."

def text_to_speech(text):
    tts = gTTS(text=text, lang="ar")
    temp_file = tempfile.NamedTemporaryFile(suffix=".mp3", delete=False)
    tts.save(temp_file.name)
    return temp_file.name

# ==============================
# 🔄 المعالجة المركزية
# ==============================

def process_interaction(audio_path):
    global points, emotion_log

    if audio_path is None:
        return "من فضلك سجل صوتك أولاً", None

    # 1. تحويل الصوت لنص (STT)
    result = whisper_model.transcribe(audio_path)
    user_text = result["text"]

    # 2. تحليل الحالة والاقتراح
    emotion = detect_emotion(user_text)
    exercise = generate_exercise(emotion)

    # 3. تحديث النقاط والسجل
    points += 10
    emotion_log.append(emotion)
    if len(emotion_log) > 5: emotion_log.pop(0)

    # 4. صياغة رد المساعد
    reply_prompt = f"أنت مساعد ذكي صديق لطفل ADHD. الطفل قال: '{user_text}' وحالته هي '{emotion}'. رد عليه بجملة تشجيعية قصيرة جداً."
    reply_text = llm.invoke(reply_prompt).content

    # 5. تجهيز المخرجات
    full_response = (
        f"🎤 أنت قلت: {user_text}\n"
        f"--- \n"
        f"🤖 الرد: {reply_text}\n\n"
        f"🎯 تمرين مقترح: {exercise}\n"
        f"⭐ نقاطك الآن: {points}\n"
        f"📊 شريط مشاعرك الأخير: {' ⬅️ '.join(emotion_log)}"
    )

    # تحويل الرد لصوت
    audio_output = text_to_speech(reply_text)

    return full_response, audio_output

# ==============================
# 🎨 واجهة Gradio (المتصفح)
# ==============================

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🧠 مساعد ADHD الذكي (نسخة الميكروفون)")
    gr.Markdown("تحدث مع المساعد وسيقوم بتحليل مشاعرك واقتراح تمارين لنشاطك.")

    with gr.Row():
        with gr.Column():
            # الميكروفون يستخدم ميكروفون المتصفح مباشرة
            audio_input = gr.Audio(
                label="اضغط للتحدث",
                type="filepath",
                sources=["microphone"]
            )
            submit_btn = gr.Button("إرسال المعالجة 🚀", variant="primary")

        with gr.Column():
            text_output = gr.Textbox(label="التحليل والرد", lines=10)
            audio_response = gr.Audio(label="رد المساعد الصوتي", autoplay=True)

    submit_btn.click(
        fn=process_interaction,
        inputs=audio_input,
        outputs=[text_output, audio_response]
    )

    gr.Examples(
        examples=[None],
        inputs=[audio_input],
        label="ملاحظة: تأكد من السماح للمتصفح بالوصول للميكروفون."
    )

# تشغيل الواجهة
demo.launch(debug=True)

/tmp/ipykernel_10802/933185069.py:99: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://7e3e96ec4fd3eb04d2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/gradio/queueing.py", line 785, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/route_utils.py", line 358, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 2162, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 1634, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/anyio/to_thread.py", line 63, in run_sync
    return await get_async_backend().run_sync_in_worker_thread(
           ^^^^^

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://7e3e96ec4fd3eb04d2.gradio.live
